# Gas Intelligence 360° · Etapa 2: Construção do Data Warehouse

Transformamos as bases tratadas em um **modelo estrela** (Kimball) dentro de um banco SQLite.
O SQLite já vem com o Python, então tudo roda no Colab sem instalar nada.

```
                dim_tempo
                    │
 dim_cidade ── fato_faturamento ── dim_contrato ── dim_cliente
      │             │
 fato_operacao  fato_ocorrencia
```

**Decisões de modelagem**
- **Chaves substitutas (`sk_`)** em todas as dimensões, independentes dos IDs do sistema de origem.
- **Membro desconhecido (`-1`)**: o contrato com cliente inexistente (CLI9999) não é descartado; ele aponta para "Cliente desconhecido". Assim nenhuma receita some do DW.
- **Fato de faturamento consolidado**: leitura, fatura e pagamento têm o mesmo grão (contrato × competência), então viram uma única fato com o status de pagamento já calculado.

In [1]:
import os, sys, subprocess, sqlite3
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/Brithysetubal/gas-intelligence-360.git"
if "google.colab" in sys.modules:
    if not Path("gas-intelligence-360").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("gas-intelligence-360")
else:
    while not Path("data/raw").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")

PROC, DW = Path("data/processed"), Path("data/dw")
DW.mkdir(parents=True, exist_ok=True)
DB_PATH = DW / "gas_dw.db"

t = {n: pd.read_csv(PROC / f"{n}_tratado.csv") for n in
     ["clientes", "contratos", "consumo", "faturas", "pagamentos", "operacoes", "ocorrencias"]}
{n: len(df) for n, df in t.items()}

{'clientes': 300, 'contratos': 300, 'consumo': 10200, 'faturas': 10200, 'pagamentos': 8876, 'operacoes': 204, 'ocorrencias': 150}

## 1. Criando a estrutura (DDL)

O script completo está em [`ddl_modelo_estrela.sql`](ddl_modelo_estrela.sql).

In [2]:
if DB_PATH.exists():
    DB_PATH.unlink()
con = sqlite3.connect(DB_PATH)
con.executescript(Path("02_data_warehouse/ddl_modelo_estrela.sql").read_text(encoding="utf-8"))
pd.read_sql("SELECT name AS tabela FROM sqlite_master WHERE type='table' ORDER BY name", con)

,tabela
0,dim_cidade
1,dim_cliente
2,dim_contrato
3,dim_tempo
4,fato_faturamento
5,fato_ocorrencia
6,fato_operacao


## 2. Dimensões

In [3]:
# dim_tempo: todas as competências presentes nas fatos
MESES = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
comps = sorted(set(t["faturas"].competencia) | set(t["operacoes"].competencia)
               | set(pd.to_datetime(t["ocorrencias"].data_ocorrencia).dt.strftime("%Y-%m")))
dim_tempo = pd.DataFrame({"competencia": comps})
dim_tempo["ano"] = dim_tempo.competencia.str[:4].astype(int)
dim_tempo["mes"] = dim_tempo.competencia.str[5:].astype(int)
dim_tempo["sk_tempo"] = dim_tempo.ano * 100 + dim_tempo.mes
dim_tempo["data_inicio_mes"] = dim_tempo.competencia + "-01"
dim_tempo["nome_mes"] = dim_tempo.mes.map(lambda m: MESES[m - 1])
dim_tempo["trimestre"] = dim_tempo.ano.astype(str) + "-T" + ((dim_tempo.mes - 1) // 3 + 1).astype(str)

# dim_cidade
cidades = sorted(set(t["clientes"].cidade) | set(t["operacoes"].cidade))
dim_cidade = pd.DataFrame({"cidade": cidades})
dim_cidade.insert(0, "sk_cidade", range(1, len(dim_cidade) + 1))
dim_cidade = pd.concat([pd.DataFrame([{"sk_cidade": -1, "cidade": "Desconhecida"}]), dim_cidade])
sk_cidade = dict(zip(dim_cidade.cidade, dim_cidade.sk_cidade))

# dim_cliente (+ membro desconhecido)
dim_cliente = t["clientes"].copy()
dim_cliente.insert(0, "sk_cliente", range(1, len(dim_cliente) + 1))
dim_cliente["sk_cidade"] = dim_cliente.cidade.map(sk_cidade)
dim_cliente["segmento_inferido"] = dim_cliente.segmento_inferido.astype(int)
desconhecido = {"sk_cliente": -1, "cliente_id": "N/A", "nome_ficticio": "Cliente desconhecido",
                "sk_cidade": -1, "segmento": "Não identificado", "status_cliente": "Desconhecido", "segmento_inferido": 0}
dim_cliente = pd.concat([pd.DataFrame([desconhecido]), dim_cliente.drop(columns="cidade")])
sk_cliente = dict(zip(dim_cliente.cliente_id, dim_cliente.sk_cliente))

# dim_contrato
dim_contrato = t["contratos"].copy()
dim_contrato.insert(0, "sk_contrato", range(1, len(dim_contrato) + 1))
dim_contrato["sk_cliente"] = dim_contrato.cliente_id.map(sk_cliente).fillna(-1).astype(int)
dim_contrato["cliente_orfao"] = dim_contrato.cliente_orfao.astype(int)
dim_contrato = dim_contrato.drop(columns="cliente_id")

for nome, df in [("dim_tempo", dim_tempo), ("dim_cidade", dim_cidade),
                 ("dim_cliente", dim_cliente), ("dim_contrato", dim_contrato)]:
    df.to_sql(nome, con, if_exists="append", index=False)
    print(f"{nome:<14} {len(df):>4} linhas")

dim_tempo        34 linhas
dim_cidade        8 linhas
dim_cliente     301 linhas
dim_contrato    300 linhas


## 3. Fatos

### 3.1 `fato_faturamento`
Regra de status de pagamento:

| Situação | Regra |
|---|---|
| Sem cobrança | fatura zerada e sem pagamento |
| Em aberto | sem pagamento válido |
| Pago parcial | valor pago menor que o faturado |
| Pago com atraso | pago integralmente após o vencimento |
| Pago no prazo | pago integralmente até o vencimento |

In [4]:
# mapa contrato → chaves de cliente e cidade
ctr = dim_contrato[["sk_contrato", "contrato_id", "sk_cliente"]].merge(
    dim_cliente[["sk_cliente", "sk_cidade"]], on="sk_cliente")

f = (t["faturas"]
     .merge(t["consumo"][["contrato_id", "competencia", "volume_m3", "tipo_leitura", "status_leitura"]],
            on=["contrato_id", "competencia"], how="left")
     .merge(t["pagamentos"][["fatura_id", "valor_pago_brl", "data_pagamento", "meio_pagamento"]],
            on="fatura_id", how="left")
     .merge(ctr, on="contrato_id", how="left"))

f["sk_tempo"] = f.competencia.str.replace("-", "").astype(int)
f["valor_pago_brl"] = f.valor_pago_brl.fillna(0)
f["dias_atraso"] = (pd.to_datetime(f.data_pagamento) - pd.to_datetime(f.data_vencimento)).dt.days

def status(r):
    if pd.isna(r.data_pagamento):
        return "Sem cobrança" if r.valor_fatura_brl == 0 else "Em aberto"
    if r.valor_pago_brl < r.valor_fatura_brl - 0.01:
        return "Pago parcial"
    return "Pago com atraso" if r.dias_atraso > 0 else "Pago no prazo"

f["status_pagamento"] = f.apply(status, axis=1)

cols = ["fatura_id", "sk_contrato", "sk_cliente", "sk_cidade", "sk_tempo", "volume_m3", "tipo_leitura",
        "status_leitura", "tarifa_brl_m3", "valor_fatura_brl", "status_fatura", "data_vencimento",
        "valor_pago_brl", "data_pagamento", "meio_pagamento", "dias_atraso", "status_pagamento"]
fato_faturamento = f[cols]
fato_faturamento.to_sql("fato_faturamento", con, if_exists="append", index=False)
fato_faturamento.status_pagamento.value_counts()

status_pagamento
Pago com atraso    7266
Em aberto          1290
Pago no prazo      1233
Pago parcial        377
Sem cobrança         34
Name: count, dtype: int64

### 3.2 `fato_operacao` e `fato_ocorrencia`

In [5]:
op = t["operacoes"].copy()
op["sk_cidade"] = op.cidade.map(sk_cidade)
op["sk_tempo"] = op.competencia.str.replace("-", "").astype(int)
fato_operacao = op[["operacao_id", "sk_cidade", "sk_tempo", "volume_distribuido_m3",
                    "volume_medido_m3", "perda_m3", "custo_operacional_brl"]]
fato_operacao.to_sql("fato_operacao", con, if_exists="append", index=False)

oc = t["ocorrencias"].merge(ctr, on="contrato_id", how="left")
oc["sk_tempo"] = pd.to_datetime(oc.data_ocorrencia).dt.strftime("%Y%m").astype(int)
fato_ocorrencia = oc[["ocorrencia_id", "sk_contrato", "sk_cliente", "sk_cidade", "sk_tempo",
                      "data_ocorrencia", "tipo_ocorrencia", "severidade", "status_ocorrencia"]]
fato_ocorrencia.to_sql("fato_ocorrencia", con, if_exists="append", index=False)
con.commit()
print("fato_operacao:", len(fato_operacao), "| fato_ocorrencia:", len(fato_ocorrencia))

fato_operacao: 204 | fato_ocorrencia: 150


## 4. Testes de validação do DW

Um DW só é confiável se for testado. Cada consulta abaixo deve retornar **0** problemas.

In [6]:
testes = {
    "Fatos sem contrato na dimensão":
        "SELECT COUNT(*) FROM fato_faturamento f LEFT JOIN dim_contrato d USING(sk_contrato) WHERE d.sk_contrato IS NULL",
    "Fatos sem competência na dim_tempo":
        "SELECT COUNT(*) FROM fato_faturamento f LEFT JOIN dim_tempo d USING(sk_tempo) WHERE d.sk_tempo IS NULL",
    "Contratos sem cliente (inclui membro -1)":
        "SELECT COUNT(*) FROM dim_contrato c LEFT JOIN dim_cliente d USING(sk_cliente) WHERE d.sk_cliente IS NULL",
    "Faturas duplicadas por contrato × mês":
        "SELECT COUNT(*) FROM (SELECT sk_contrato, sk_tempo FROM fato_faturamento GROUP BY 1,2 HAVING COUNT(*) > 1)",
    "Valores de fatura nulos":
        "SELECT COUNT(*) FROM fato_faturamento WHERE valor_fatura_brl IS NULL",
    "Operações sem cidade":
        "SELECT COUNT(*) FROM fato_operacao WHERE sk_cidade IS NULL",
}
resultado = pd.DataFrame([{"teste": k, "problemas": con.execute(q).fetchone()[0]} for k, q in testes.items()])
resultado["ok"] = resultado.problemas.eq(0).map({True: "✅", False: "❌"})
assert resultado.problemas.sum() == 0, "Há falhas de integridade no DW"
resultado

,teste,problemas,ok
0,Fatos sem contrato na dimensão,0,✅
1,Fatos sem competência na dim_tempo,0,✅
2,Contratos sem cliente (inclui membro -1),0,✅
3,Faturas duplicadas por contrato × mês,0,✅
4,Valores de fatura nulos,0,✅
5,Operações sem cidade,0,✅


In [7]:
# Reconciliação: o total faturado no DW precisa bater com a base tratada
total_dw = con.execute("SELECT ROUND(SUM(valor_fatura_brl), 2) FROM fato_faturamento").fetchone()[0]
total_csv = round(t["faturas"].valor_fatura_brl.sum(), 2)
print(f"Faturado no DW: R$ {total_dw:,.2f} | Base tratada: R$ {total_csv:,.2f} | Bate? {total_dw == total_csv}")

Faturado no DW: R$ 11,927,488.34 | Base tratada: R$ 11,927,488.34 | Bate? True


## 5. Exportando para o Power BI

O Power BI lê CSV nativamente; exportamos cada tabela do DW para `data/dw/`.

In [8]:
for (tabela,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall():
    pd.read_sql(f"SELECT * FROM {tabela}", con).to_csv(DW / f"{tabela}.csv", index=False)
con.close()
sorted(p.name for p in DW.iterdir())

['dim_cidade.csv', 'dim_cliente.csv', 'dim_contrato.csv', 'dim_tempo.csv', 'fato_faturamento.csv', 'fato_ocorrencia.csv', 'fato_operacao.csv', 'gas_dw.db']

✅ DW pronto em `data/dw/gas_dw.db`. Próxima etapa: **[03 · Análises em SQL](../03_sql/03_analises_sql.ipynb)**.